# 00 - How does noise become a new sample?

The series follows one generator from training to post-hoc steering. We begin
with the pieces that are easiest to confuse: the data distribution, random
starting noise, the neural prediction, and the numerical sampler. In two
dimensions all four can be drawn before we meet the same ideas in images.

The model is unconditional: class labels are never given to its velocity
network. Later notes use labels only to construct and evaluate controls after
training.

**Time convention for Notes 00--05:** `t=0` is noise and `t=1` is data.


In [ ]:
import sys
from pathlib import Path

NOTES_DIR = Path.cwd()
if not (NOTES_DIR / "toy_notes.py").exists():
    NOTES_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(NOTES_DIR.resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_notes import *
from notebook_views import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print("device:", device)


## 1. What is a generative model?

A generative model learns a **sampling procedure** from examples. A useful
generator produces new samples with the structure of the training distribution
rather than returning a stored example.

For the flow model in these notes:

- an easy source distribution supplies fresh Gaussian noise;
- a neural network learns a time-dependent velocity;
- an ODE sampler follows that velocity from noise to a generated endpoint.

The neural network alone is not the complete generator. The source
distribution, trained model, and sampler work together.

**Before you run:** In the diagram, which object is learned from data? Which
object supplies the variation between generated samples?


In [ ]:
plot_generator_pipeline()


The velocity network is learned. Fresh noise supplies different starting
states, while the sampler turns each starting state into one endpoint.

### Three ways to request control

- An **unconditional** generator models all training examples without receiving
  a requested class. That is the model trained here.
- A **conditional** generator receives extra information, such as a class label
  or text prompt, during training and generation.
- **Post-hoc steering** keeps a trained generator frozen and modifies the
  sampling computation. Notebooks `03`--`05` study this third setting.

Post-hoc steering is the setting used later: the unconditional model remains
frozen while the sampling computation is changed. A deterministic sampler can
still generate a diverse collection because different initial noise points
follow different deterministic trajectories.


## 2. What information does noise hide?

A probability distribution describes which regions are likely and how often
different kinds of samples occur. Images live in thousands or millions of
dimensions, where that structure is difficult to draw. Here every sample has
only two coordinates, so a large scatter plot makes cluster locations,
frequencies, shapes, and spread directly visible. The three colors play the
role of mixture-component labels. Later notes treat those labels as toy classes
for steering and evaluation. The velocity model will not receive them.


In [ ]:
clean, labels = sample_labeled_mixture(2400, device=device)
fig, ax = plt.subplots(figsize=(6, 5))
plot_labeled_points(clean, labels, ax=ax, title="Clean toy data")
ax.legend(frameon=False)
plt.show()


## 3. One path, read in two directions

First sample base noise, then use the linear path

$$x_{noise}=s\epsilon,\qquad \epsilon\sim\mathcal N(0,I),$$

$$x_t=(1-t)x_{noise}+t x_{data},\qquad s=1.8.$$

- At `t=1`, the point is clean data.
- At `t=0`, the point is Gaussian noise.
- Lower `t` therefore means higher noise in this notebook series.

A diffusion model learns to undo a noising process. Flow matching learns the
velocity along a path between the same endpoints.

### Diffusion and flow matching in one table

| Family | Typical learned quantity | Generation step |
|---|---|---|
| Diffusion model | noise, score, or a denoised estimate | repeatedly remove noise with a stochastic or deterministic sampler |
| Flow-matching model | velocity along a chosen probability path | integrate an ODE from the source distribution to data |

Both are time-dependent generative models. Here `t` indexes a position along
the noise-to-data path; it is not elapsed computer time.

**Before you run:** As `t` increases from noise to data, which structure should
become visible first: the exact identity of every point, or the coarse cluster
layout? The same clean points and noise points are reused in every panel.


In [ ]:
set_seed(SEED)
clean_small, labels_small = sample_labeled_mixture(1500, device=device)
fixed_noise = sample_base(clean_small.shape[0], device=device)

times = [0.08, 0.35, 0.70, 1.0]
fig, axes = plt.subplots(1, len(times), figsize=(15, 3.8))
for ax, t_value in zip(axes, times):
    t = torch.full((clean_small.shape[0], 1), t_value, device=device)
    noisy = (1 - t) * fixed_noise + t * clean_small
    plot_labeled_points(noisy, labels_small, ax=ax, title=f"t={t_value:.2f}", alpha=0.42)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
plt.tight_layout()
plt.show()


The coarse three-cluster layout appears before individual points reach their
clean endpoints. This early-to-late change in available information will matter
when we choose a steering window.

**Change one thing:** Replace `0.35` in `times` with another value between zero
and one. Rerun only the preceding cell and locate when the clusters first become
easy to distinguish.

## 4. What flow matching predicts

Choose a noise point `x_noise`, a data point `x_data`, and a time `t`. Along the
straight conditional path, the target velocity is simply

$$u_t = x_{data} - x_{noise}.$$

Training repeatedly asks a neural network to predict this arrow from only
`(x_t, t)`. Because many endpoint pairs can pass near the same `x_t`, the
network learns an average velocity field, not a memorized arrow for each pair.
The pair-specific target is available while constructing training examples but
is not available during generation.

**Before you run:** The arrows below come from individual endpoint pairs. If
two different pairs pass through nearly the same state, can the network recover
both pair-specific arrows from `(x_t,t)` alone?


In [ ]:
set_seed(SEED + 1)
x_data, _ = sample_labeled_mixture(24, device=device)
x_noise = sample_base(24, device=device)
t_value = 0.42
t = torch.full((24, 1), t_value, device=device)
x_t = (1 - t) * x_noise + t * x_data
target_velocity = x_data - x_noise

plot_flow_matching_batch(x_noise, x_data, x_t, target_velocity)


No. The network sees the state and time, not the hidden endpoint pair. Under
mean-squared-error training it learns the conditional average of compatible
pair velocities. Notebook `01` tests whether those local averages assemble into
a useful global flow.

## 5. Why keep the model and sampler separate?

Later we will change the numerical solver and add steering terms. Those changes
are easier to reason about if we know which part was learned and which part was
chosen at sampling time.

| Object | Job | Learned? |
|---|---|---|
| Training dataset | Supplies examples of the distribution to imitate | No |
| Base distribution | Supplies fresh, easy-to-sample Gaussian noise | No |
| Noising path | Defines intermediate training states | Chosen by us |
| Neural model | Predicts the local velocity `v_theta(t,x_t)` | Yes |
| Denoiser view | Converts velocity into a clean-data estimate | Derived from the model |
| Sampler | Numerically follows model predictions | Chosen by us |

In these notes, **generator** refers to the complete sampling procedure: base
distribution + trained neural model + sampler. Replacing Euler with RK4 changes
only the sampler; the source distribution and learned velocity field remain the
same.

Notebook `01` now supplies the missing piece: train the velocity field and ask
whether accurate local predictions actually produce the whole data
distribution.
